# XGBoost


In [11]:
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

In [2]:
base_path = Path("..")

processed_path = base_path / "data" / "processed"
features_path = processed_path / "features"
train_features_path = features_path / "train_features.csv"
test_features_path = features_path / "test_features.csv"
val_features_path = features_path / "val_features.csv"

In [3]:
train_df = pd.read_csv(train_features_path)
val_df = pd.read_csv(val_features_path)
test_df = pd.read_csv(test_features_path)

train_df.shape, val_df.shape, test_df.shape

((2415, 60), (515, 60), (520, 60))

In [4]:
target_cols = [
    "room_size_norm",
    "wet_level_norm",
    "rate_hz_norm",
    "depth_norm"
]

meta_cols = [
    "source_id",
    "wet_path"
]

In [10]:
drop_cols = target_cols + meta_cols

X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)
X_val = val_df.drop(columns=drop_cols)

y_train = train_df[target_cols]
y_test = test_df[target_cols]
y_val = val_df[target_cols]

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(2415, 54) (2415, 4)
(515, 54) (515, 4)
(520, 54) (520, 4)


## Первая модель

In [12]:
base_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model = MultiOutputRegressor(base_model)

In [13]:
model.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,"XGBRegressor(...ree=None, ...)"
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[XGBRegressor(...ree=None, ...), XGBRegressor(...ree=None, ...), XGBRegressor(...ree=None, ...), XGBRegressor(...ree=None, ...)]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[<U16](54,)","['rms_mean','rms_std','rms_median',...,'mfcc_13_mean','mfcc_13_std', 'mfcc_13_median']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,54
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None


In [15]:
y_val_pred = model.predict(X_val)
y_val_pred.shape

(515, 4)

In [ ]:
def scores(y_true, y_pred, target_cols):
    results = []

    for i, target in enumerate(target_cols):
        mae = mean_absolute_error(
            y_true[target],
            y_pred[:, i]
        )

        rmse = mean_squared_error(
        y_true[target],
        y_pred[:, i]
        ) ** 0.5

        r2 = r2_score(
        y_true[target],
        y_pred[:, i]
        )

        results.append({
            "target": target,
            "mae": mae,
            "rmse": rmse,
            "r2": r2
        })

    return pd.DataFrame(results)


val_scores = scores(y_val, y_val_pred, target_cols)


print(val_scores)

mean_mae = val_scores["mae"].mean()
mean_r2 = val_scores["r2"].mean()

print("Mean MAE:", mean_mae)
print("Mean R^2:", mean_r2)


           target       mae      rmse        r2
0  room_size_norm  0.184336  0.229835  0.382276
1  wet_level_norm  0.186729  0.226672  0.366472
2    rate_hz_norm  0.221273  0.271307  0.150986
3      depth_norm  0.158746  0.202073  0.508668
Mean MAE: 0.18777090898822102
Mean R^2: 0.352100295869633


#### Проверка на переобучение

In [25]:
y_train_predict = model.predict(X_train)

train_scores = scores(y_train, y_train_predict, target_cols)
train_scores

,target,mae,rmse,r2
0,room_size_norm,0.047030,0.062276,0.952588
1,wet_level_norm,0.044216,0.057668,0.959496
2,rate_hz_norm,0.053582,0.069329,0.941860
3,depth_norm,0.038194,0.050529,0.969655


Модель переобучилась

## Вторая модель